In [ ]:
# fmt: off
%matplotlib inline
%config InlineBackend.figure_format = "retina"
%load_ext autoreload
%load_ext jupyter_black
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pickle
from mls_scf_tools.mls_pint import ureg
plt.style.use("mls_scf")
def vdir(obj):
    return [x for x in dir(obj) if not x.startswith('__')]
import IPython
# if IPython.__version__ != "9.4.0":
#    raise UserWarning("Perhaps we don't need the ipython work around any more")
from IPython.extensions.deduperreload.deduperreload import DeduperReloader
DeduperReloader.enabled = False
# fmt: on

In [ ]:
from __future__ import annotations
from typing import Optional
import math

import numpy as np
from PIL import Image

# 8.5x11 portrait aspect ratio (height / width)
ASPECT_8_5x11 = 11.0 / 8.5

In [ ]:
def process_image_to_8_5x11(
    input_path: str,
    output_path: str,
    trim_top_px: int,
    trim_bottom_px: int,
    fade_height_px: int,
    aspect_ratio: float = ASPECT_8_5x11,
) -> None:
    """
    Read a JPEG, trim top/bottom, fade-to-black at the top, and pad with black
    above to reach a given aspect ratio (default: 8.5x11 portrait) without
    changing the horizontal resolution.

    Steps:
      1. Load image and convert to RGB.
      2. Trim `trim_top_px` rows from the top and `trim_bottom_px` rows from the bottom.
      3. Apply a vertical fade-to-black over `fade_height_px` rows at the top of the
         trimmed image (inside the image, not the padding).
      4. Compute target height from width and `aspect_ratio` and pad *above* with
         full-black rows to reach that height (if possible).

    Notes:
      - If the trimmed image is already taller than the target height, no padding
        is added; the image is left as-is (you could add extra logic to crop).
      - All units are pixels.
    """
    # --- Load and convert to RGB array ---
    img = Image.open(input_path).convert("RGB")
    arr = np.array(img)  # shape (H, W, 3)
    H, W, C = arr.shape

    # --- Trim top and bottom ---
    if trim_top_px < 0 or trim_bottom_px < 0:
        raise ValueError("trim_top_px and trim_bottom_px must be non-negative.")

    new_top = trim_top_px
    new_bottom = H - trim_bottom_px

    if new_bottom <= new_top:
        raise ValueError(
            f"Trimming removes all pixels: original height={H}, "
            f"trim_top_px={trim_top_px}, trim_bottom_px={trim_bottom_px}"
        )

    arr = arr[new_top:new_bottom, :, :]
    H_trimmed, W_trimmed, _ = arr.shape

    # --- Fade-to-black at the top of the trimmed image ---
    # Fade height can't exceed the image height.
    if fade_height_px > 0:
        fh = min(fade_height_px, H_trimmed)

        # alpha goes from 0 at the very top row (black) to ~1 at bottom of fade.
        # Using endpoint=False avoids the last row being exactly 1; you can change
        # to endpoint=True if you want that.
        alpha = np.linspace(0.0, 1.0, fh, endpoint=False, dtype=np.float32)
        alpha = alpha[:, None, None]  # shape (fh, 1, 1) for broadcasting

        top_region = arr[:fh].astype(np.float32)
        faded_region = (top_region * alpha).astype(np.uint8)
        arr[:fh] = faded_region

    # --- Compute target height from aspect ratio and pad with black above ---
    # Keep width fixed; determine desired height for given aspect ratio.
    target_height = int(math.ceil(W_trimmed * aspect_ratio))

    pad_rows = max(0, target_height - H_trimmed)
    if pad_rows > 0:
        black_pad = np.zeros((pad_rows, W_trimmed, 3), dtype=np.uint8)
        arr_out = np.vstack([black_pad, arr])
    else:
        # Already tall enough (or taller) – leave as-is.
        arr_out = arr

    # --- Save result ---
    out_img = Image.fromarray(arr_out, mode="RGB")
    out_img.save(output_path, quality=95)

In [ ]:
process_image_to_8_5x11(
    input_path="iss-limb-original.jpg",
    output_path="iss-limb.jpg",
    trim_bottom_px=1080,
    trim_top_px=200,
    fade_height_px=200,
)